<a href="https://colab.research.google.com/github/MissingSunshine/DZSpider/blob/master/econ6083/final-project/notebooks/option3_did.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Option 3: Difference-in-Differences
**Replication**: Callaway & Sant'Anna (2021) - Difference-in-Differences with Multiple Time Periods
**Data**: Minimum Wage and Employment (Card & Krueger, 1994; Dube et al., 2010)

**Key Question**: Do minimum wage increases reduce employment?

## 1. Setup and Data Download

In [ ]:
# Install required packages
!pip install -q pandas numpy matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Download minimum wage data
# Using Card & Krueger (1994) data on fast-food employment in NJ and PA
url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/Ecdat/MinimumWage.csv"

# Alternative: Create simulated staggered DiD data similar to Dube et al.
np.random.seed(42)

# Simulate state-level panel data with staggered treatment
n_states = 50
n_periods = 20
states = [f'State_{i:02d}' for i in range(n_states)]
periods = list(range(2000, 2000 + n_periods))

# Treatment timing: staggered adoption between 2005-2012
treat_time = np.random.choice([2005, 2006, 2007, 2008, 2009, 2010, np.nan],
                              size=n_states, p=[0.15, 0.15, 0.15, 0.15, 0.15, 0.15, 0.1])

data = []
for i, state in enumerate(states):
    for t in periods:
        treated = 0 if np.isnan(treat_time[i]) else (t >= treat_time[i])
        # Base employment with trend
        base = 100 + 2*(t - 2000) + np.random.normal(0, 5)
        # Treatment effect (positive in this simulation)
        effect = 5 * treated  # Minimum wage increases employment slightly
        # State fixed effect
        state_fe = np.random.normal(0, 10)
        employment = base + effect + state_fe

        data.append({
            'state': state,
            'year': t,
            'employment': employment,
            'treated': int(treated),
            'treat_time': treat_time[i]
        })

df = pd.DataFrame(data)

print(f"Dataset shape: {df.shape}")
print(f"\nStates: {df["state"].nunique()}")
print(f"Years: {df["year"].min()} - {df["year"].max()}")
print(f"\nTreatment adoption by year:")
print(df[df["treat_time"].notna()]["treat_time"].value_counts().sort_index())
df.head()

## 2. Data Exploration

In [ ]:
# Visualize treatment adoption
treat_counts = df[df["treat_time"].notna()].groupby("treat_time")["state"].nunique()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
treat_counts.plot(kind='bar')
plt.xlabel('Treatment Year')
plt.ylabel('Number of States')
plt.title('Staggered Treatment Adoption')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)

# Plot employment trends by treatment status
plt.subplot(1, 2, 2)
treated_states = df[df["treat_time"].notna()]["state"].unique()
control_states = df[df["treat_time"].isna()]["state"].unique()

trend_treated = df[df["state"].isin(treated_states)].groupby("year")["employment"].mean()
trend_control = df[df["state"].isin(control_states)].groupby("year")["employment"].mean()

plt.plot(trend_treated.index, trend_treated.values, label='Treated States', marker='o')
plt.plot(trend_control.index, trend_control.values, label='Control States', marker='s')
plt.xlabel('Year')
plt.ylabel('Employment')
plt.title('Employment Trends')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. DiD Implementation

In [ ]:
def compute_att_gt(df, group_col='state', time_col='year', outcome='employment', treatment='treated'):
    """
    Compute ATT(g,t) following Callaway & Sant'Anna (2021)

    Group g: units first treated at time g
    Compare to never-treated or not-yet-treated units

    ATT(g,t) = [E[Y_t|g] - E[Y_g|g]] - [E[Y_t|control] - E[Y_g|control]]

    Returns: DataFrame with ATT(g,t) estimates
    """
    results = []

    # Get unique groups and time periods
    groups = sorted(df[df["treat_time"].notna()]["treat_time"].unique())
    times = sorted(df[time_col].unique())

    for g in groups:
        # Units first treated at time g
        treated_units = df[df["treat_time"] == g]

        # Control: never-treated units
        control_units = df[df["treat_time"].isna()]

        for t in times:
            if t < g:
                continue  # Pre-treatment

            # DID: (Y_t - Y_g) for treated vs control
            y_t_treat = treated_units[treated_units[time_col] == t][outcome].mean()
            y_g_treat = treated_units[treated_units[time_col] == g][outcome].mean()
            y_t_ctrl = control_units[control_units[time_col] == t][outcome].mean()
            y_g_ctrl = control_units[control_units[time_col] == g][outcome].mean()

            if pd.notna(y_t_treat) and pd.notna(y_g_treat) and pd.notna(y_t_ctrl) and pd.notna(y_g_ctrl):
                att_gt = (y_t_treat - y_g_treat) - (y_t_ctrl - y_g_ctrl)
                results.append({'group': g, 'time': t, 'att_gt': att_gt})

    return pd.DataFrame(results)

def event_study_plot(att_gt_df):
    """
    Plot dynamic treatment effects by event time
    """
    # Calculate event time (time - group)
    att_gt_df = att_gt_df.copy()
    att_gt_df['event_time'] = att_gt_df['time'] - att_gt_df['group']

    # Aggregate by event time
    event_study = att_gt_df.groupby('event_time')['att_gt'].agg(['mean', 'std', 'count'])
    event_study = event_study[event_study['count'] >= 3]  # Require at least 3 groups
    event_study['ci_lower'] = event_study['mean'] - 1.96 * event_study['std'] / np.sqrt(event_study['count'])
    event_study['ci_upper'] = event_study['mean'] + 1.96 * event_study['std'] / np.sqrt(event_study['count'])

    # Plot
    plt.figure(figsize=(10, 6))
    plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    plt.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='Treatment')
    plt.plot(event_study.index, event_study['mean'], 'o-', color='blue', markersize=8)
    plt.fill_between(event_study.index, event_study['ci_lower'], event_study['ci_upper'],
                     alpha=0.2, color='blue')
    plt.xlabel('Time Relative to Treatment')
    plt.ylabel('ATT(g,t)')
    plt.title('Event Study: Dynamic Treatment Effects')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    return event_study

## 4. Run Analysis

In [ ]:
# Compute ATT(g,t)
att_results = compute_att_gt(df)

print("ATT(g,t) Results:")
print(att_results.head(10))

# Plot event study
event_study = event_study_plot(att_results)

print("\nAverage Effect by Event Time:")
print(event_study[['mean', 'ci_lower', 'ci_upper']].round(2))

## 5. Two-Way Fixed Effects (TWFE) Comparison

In [ ]:
# Traditional TWFE regression
import statsmodels.formula.api as smf

# Create state and year fixed effects
df['post'] = (df['year'] >= df['treat_time']).astype(int)
df.loc[df["treat_time"].isna(), "post"] = 0

# TWFE regression
twfe = smf.ols('employment ~ treated + C(state) + C(year)', data=df).fit()

print("Traditional TWFE Results:")
print(f"Coefficient on treated: {twfe.params["treated"]:.3f}")
print(f"Standard error: {twfe.bse["treated"]:.3f}")
print(f"t-statistic: {twfe.tvalues["treated"]:.3f}")

print("\n" + "="*60)
print("Key Finding:")
print("The event study shows dynamic treatment effects over time.")
print("Pre-treatment coefficients should be close to zero (parallel trends).")
print("Post-treatment coefficients show the causal effect.")
print("="*60)

## Summary

This notebook implements staggered DiD following Callaway & Sant'Anna (2021):

1. **ATT(g,t)**: Group-time average treatment effects
2. **Event Study**: Dynamic effects over time
3. **Parallel Trends**: Check pre-treatment coefficients
4. **TWFE Comparison**: Compare with traditional approach

**Extensions to try**:
- Compare TWFE vs Callaway-Sant'Anna vs Sun-Abraham vs Borusyak
- Test sensitivity to donor pool composition
- Implement Doubly Robust DiD
- Use real Card-Krueger or QCEW data

# 导入第三方库

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import statsmodels.formula.api as smf
import warnings
import os
from scipy.stats.mstats import winsorize
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy import stats

warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Deja Vu Sans']
plt.rcParams['axes.unicode_minus'] = False
rcParams.update({'figure.figsize': (12, 8), 'font.size': 12})

OUTPUT_DIR = os.path.join(os.getcwd(),"output")

if not os.path.exists(OUTPUT_DIR):
    os.mkdir(OUTPUT_DIR)

In [4]:
df = pd.read_csv('https://raw.githubusercontent.com/JasmineHao/JasmineHao.github.io/main/econ6083/final-project/notebooks/data/china_city_panel_with_policies.csv')
df.head()

,year,province,city,prov_code,city_code,region,hu_line,gdp,gdp_per_capita,gdp_growth,...,sponge_first_year,cbec_treat,cbec_batch,cbec_first_year,ltci_treat,ltci_batch,ltci_first_year,smart_treat,smart_batch,smart_first_year
0,1990,北京市,北京市,110000,110000,东部,东南侧,5008200.0,4883.0,NaN,...,0,1,2,2018,1,2,2020,1,3,2015
1,1991,北京市,北京市,110000,110000,东部,东南侧,5586900.0,5395.0,NaN,...,0,1,2,2018,1,2,2020,1,3,2015
2,1992,北京市,北京市,110000,110000,东部,东南侧,7334726.0,7410.0,NaN,...,0,1,2,2018,1,2,2020,1,3,2015
3,1993,北京市,北京市,110000,110000,东部,东南侧,9082551.0,9424.0,NaN,...,0,1,2,2018,1,2,2020,1,3,2015
4,1994,北京市,北京市,110000,110000,东部,东南侧,10830377.0,11439.0,NaN,...,0,1,2,2018,1,2,2020,1,3,2015


In [5]:

core_vars = ['city_code', 'year', 'region', 'sponge_treat', 'so2_emission', 'pm25', 'gdp', 'pop_avg', 'fiscal_revenue', 'fiscal_exp', 'secondary_share', 'tertiary_share', 'edu_exp', 'sci_exp']
df = df[core_vars].copy()

df = df[df['so2_emission'] > 0].copy()
df['ln_so2'] = np.log(df['so2_emission'])
df['ln_gdp'] = np.log(df['gdp'])
df['ln_pop'] = np.log(df['pop_avg'])
df['ln_fiscal'] = np.log(df['fiscal_exp'])
df['ln_edu'] = np.log(df['edu_exp'] + 1)
df['gdp_per_capita'] = df['gdp'] / df['pop_avg']
df['ln_gdp_per_capita'] = np.log(df['gdp_per_capita'])

df['ln_so2'] = winsorize(df['ln_so2'], limits=[0.01, 0.01])
df['pm25'] = winsorize(df['pm25'], limits=[0.01, 0.01])
df['ln_gdp'] = winsorize(df['ln_gdp'], limits=[0.01, 0.01])
df['ln_pop'] = winsorize(df['ln_pop'], limits=[0.01, 0.01])

df['treat_year'] = df.groupby('city_code')['year'].transform(lambda x: x[df.loc[x.index, 'sponge_treat'] == 1].min() if (df.loc[x.index, 'sponge_treat'] == 1).any() else np.nan)
df['treated_group'] = df['treat_year'].notna().astype(int)
df = df.sort_values(['city_code', 'year']).reset_index(drop=True)
df['event_time'] = df['year'] - df['treat_year']
df['event_time'] = df['event_time'].fillna(-99)
df['year_trend'] = df['year'] - 2002
df['year_trend_sq'] = df['year_trend'] ** 2

df['city_size'] = pd.cut(df['pop_avg'], bins=[0, 500, 1000, np.inf], labels=['small', 'medium', 'large'])
df['economy_level'] = pd.qcut(df['gdp_per_capita'], q=3, labels=['low', 'medium', 'high'])
df['industry_structure'] = pd.qcut(df['secondary_share'], q=3, labels=['low_secondary', 'mid_secondary', 'high_secondary'])

df_analysis = df.dropna(subset=['ln_so2', 'sponge_treat', 'ln_gdp', 'ln_pop', 'ln_fiscal', 'secondary_share', 'tertiary_share', 'ln_edu', 'city_code', 'year']).copy()
df_analysis.head()

,city_code,year,region,sponge_treat,so2_emission,pm25,gdp,pop_avg,fiscal_revenue,fiscal_exp,...,gdp_per_capita,ln_gdp_per_capita,treat_year,treated_group,event_time,year_trend,year_trend_sq,city_size,economy_level,industry_structure
0,110000,2003,东部,0,114012.0,NaN,36631000.0,1144.49,5925388.0,7348043.0,...,32006.395862,10.373691,NaN,0,-99.0,1,1,large,medium,low_secondary
1,110000,2004,东部,0,114012.0,NaN,42833100.0,1159.68,7444874.0,8982756.0,...,36935.275248,10.516922,NaN,0,-99.0,2,4,large,medium,low_secondary
2,110000,2005,东部,0,105473.0,NaN,68863101.0,1174.88,9192098.0,10583114.0,...,58612.880464,10.978710,NaN,0,-99.0,3,9,large,high,low_secondary
3,110000,2006,东部,0,93755.0,NaN,78702835.0,1190.07,11171514.0,12968389.0,...,66132.945961,11.099422,NaN,0,-99.0,4,16,large,high,low_secondary
4,110000,2007,东部,0,82909.0,NaN,93533200.0,1205.26,14926380.0,16495023.0,...,77604.168395,11.259376,NaN,0,-99.0,5,25,large,high,low_secondary


In [6]:
print("="*80)
print(f"数据维度: {df.shape} | 城市数量: {df['city_code'].nunique()} | 年份范围: {df['year'].min()}-{df['year'].max()}")
print(f"处理组城市数: {df[df['treated_group']==1]['city_code'].nunique()} | 对照组城市数: {df[df['treated_group']==0]['city_code'].nunique()}")
print("政策实施年份分布：")
print(df[df['treat_year'].notna()]['treat_year'].value_counts().sort_index())
print("区域分布：")
print(df['region'].value_counts())
print("城市规模分布：")
print(df['city_size'].value_counts())
print("="*80)


数据维度: (6059, 29) | 城市数量: 299 | 年份范围: 2003-2023
处理组城市数: 23 | 对照组城市数: 276
政策实施年份分布：
treat_year
2003.0    477
Name: count, dtype: int64
区域分布：
region
东部    2113
中部    2096
西部    1850
Name: count, dtype: int64
城市规模分布：
city_size
small     3190
medium    1456
large      175
Name: count, dtype: int64


In [8]:
summary_full = df[['ln_so2', 'pm25', 'ln_gdp', 'ln_pop', 'ln_fiscal', 'secondary_share', 'tertiary_share', 'ln_edu', 'gdp_per_capita']].describe().T.round(4)
summary_full.head()

,count,mean,std,min,25%,50%,75%,max
ln_so2,6059.0,9.8857,1.4046,5.6836,8.9286,10.0758,10.9907,12.3557
pm25,1912.0,35.3964,12.4273,17.0000,26.0000,34.0000,43.0000,116.0000
ln_gdp,6059.0,16.2185,1.1106,13.7367,15.4527,16.2107,16.9510,19.0555
ln_pop,4821.0,5.8583,0.6877,3.9086,5.4638,5.9128,6.3464,8.1345
ln_fiscal,6059.0,14.3645,1.1419,10.4058,13.6007,14.5013,15.1252,18.3581
